
# 06 – Iteration 3: Product Pipeline Demo

This notebook demonstrates the **end-to-end anomaly scoring pipeline** for Dataset 2
(factures amb consums modificats) using the assets produced in previous notebooks:

- Cleaned dataset (Notebook 01)
- Feature engineering logic (Notebook 02 / `src/features.py`)
- Preprocessor parameters saved in `preprocessor_iter3_params.json` (Notebook 04)
- Trained RandomForest model from Notebook 05

Pipeline shown here:

1. Load the cleaned Dataset 2 (iteration 3 version)
2. Take a small random demo sample
3. Build engineered features on top of the raw fields
4. Select the same feature columns used for model training
5. Reconstruct the preprocessor (imputer + scaler) from saved params
6. Use the trained model to obtain anomaly scores
7. Show the top-N most suspicious records


In [ ]:

import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

# Allow importing project src/
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from features import build_features  # feature engineering used in Notebook 2

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

RESULTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results"))
CLEANED_DIR = os.path.join(RESULTS_DIR, "cleaned")
PREPARED_DIR = os.path.join(RESULTS_DIR, "prepared")
MODELS_DIR = os.path.join(RESULTS_DIR, "models")

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RESULTS_DIR  :", RESULTS_DIR)
print("CLEANED_DIR  :", CLEANED_DIR)
print("PREPARED_DIR :", PREPARED_DIR)
print("MODELS_DIR   :", MODELS_DIR)


In [ ]:

# 1) Load preprocessor parameters saved in Notebook 04
PARAMS_PATH = os.path.join(PREPARED_DIR, "preprocessor_iter3_params.json")

if not os.path.exists(PARAMS_PATH):
    raise FileNotFoundError(f"Preprocessor params JSON not found: {PARAMS_PATH}")

with open(PARAMS_PATH, "r") as f:
    params = json.load(f)

feature_cols = params["feature_cols"]
imputer_strategy = params.get("imputer_strategy", "median")
scaler_mean = np.array(params["scaler_mean"], dtype=float)
scaler_scale = np.array(params["scaler_scale"], dtype=float)

# Rebuild scaler
scaler = StandardScaler()
scaler.mean_ = scaler_mean
scaler.scale_ = scaler_scale
scaler.n_features_in_ = scaler_mean.shape[0]

# Imputer: we might not have stored statistics, so we handle both cases
imputer_statistics = params.get("imputer_statistics", None)
if imputer_statistics is not None:
    imputer = SimpleImputer(strategy=imputer_strategy)
    imputer.statistics_ = np.array(imputer_statistics, dtype=float)
    print("Reconstructed imputer from saved statistics.")
else:
    # We will fit the imputer later on the demo data if needed
    imputer = None
    print("No imputer statistics in params. Will fit imputer on demo data if needed.")

print("\nLoaded preprocessor params from:", PARAMS_PATH)
print(" - #feature_cols:", len(feature_cols))
print(" - imputer_strategy:", imputer_strategy)


In [ ]:

# 2) Load trained model
MODEL_PATH = os.path.join(MODELS_DIR, "model_iter3_rf.joblib")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

model = joblib.load(MODEL_PATH)
print("Loaded model from:", MODEL_PATH)
print("Model type:", type(model))


In [ ]:

# 3) Load cleaned Dataset 2 (iteration 3)
CLEANED_PARQUET_PATH = os.path.join(CLEANED_DIR, "dataset2_cleaned_iter3.parquet")

if not os.path.exists(CLEANED_PARQUET_PATH):
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_PARQUET_PATH}")

df_clean_full = pd.read_parquet(CLEANED_PARQUET_PATH)
print("Full cleaned dataset shape:", df_clean_full.shape)

# Take a small random sample for the demo
n_demo = 1000
if df_clean_full.shape[0] > n_demo:
    df_clean_demo = df_clean_full.sample(n=n_demo, random_state=42)
else:
    df_clean_demo = df_clean_full.copy()

print("Demo subset shape:", df_clean_demo.shape)
df_clean_demo.head()


In [ ]:

# 4) Build engineered features for the demo subset
df_feat_demo = build_features(df_clean_demo)
print("Feature-augmented demo shape:", df_feat_demo.shape)
df_feat_demo.head()


In [ ]:

# 5) Select the same feature columns used during training
missing_feats = [c for c in feature_cols if c not in df_feat_demo.columns]
if missing_feats:
    print("WARNING: The following expected feature columns are missing in the demo feature table:")
    print(missing_feats)

used_features = [c for c in feature_cols if c in df_feat_demo.columns]
print("Using", len(used_features), "features for the demo:")
print(used_features)

X_demo_raw = df_feat_demo[used_features].values

# If we do not have saved imputer statistics, fit imputer on demo data
if imputer is None:
    imputer = SimpleImputer(strategy=imputer_strategy)
    imputer.fit(X_demo_raw)
    print("Fitted a new imputer on demo data.")

X_demo_imp = imputer.transform(X_demo_raw)
X_demo = scaler.transform(X_demo_imp)

print("X_demo shape:", X_demo.shape)


In [ ]:

# 6) Get anomaly scores (probability of y_anom = 1)
if hasattr(model, "predict_proba"):
    scores = model.predict_proba(X_demo)[:, 1]
else:
    scores = model.decision_function(X_demo)

df_out = df_feat_demo.copy()
df_out["anom_score"] = scores

key_cols = [
    "POLISSA_SUBM",
    "NUMEROSERIECONTADOR",
    "FECHA_HORA",
    "CODI_ANOMALIA",
    "y_anom",
    "US_AIGUA_SUBM",
    "SECCIO_CENSAL",
]
key_cols = [c for c in key_cols if c in df_out.columns]

display_cols = key_cols + ["anom_score"]

df_out_sorted = df_out.sort_values("anom_score", ascending=False)

print("Top 20 most suspicious records (by model score):")
df_out_sorted[display_cols].head(20)


In [ ]:

print("Demo pipeline summary:")
print("- Input rows (demo):", df_clean_demo.shape[0])
print("- Engineered features used:", len(used_features))
print("- Anomaly score range:",
      float(df_out['anom_score'].min()),
      "to",
      float(df_out['anom_score'].max()))
